# Business entity resolution EDA
Uses the same setup/configuration as the hybrid baseline. It first proves file integrity and ground-truth ID coverage, then runs EDA. Set EDA_MAX_ROWS to 0 for a full scan, or a positive limit for an explicitly labeled prefix sample.


In [ ]:
from pathlib import Path
import subprocess, sys, json, shutil, hashlib
REPO = Path('/content/Amazon-ML-Challange-2026')
URL = 'https://github.com/alisalmann7386-crypto/Amazon-ML-Challange-2026.git'
if not REPO.exists():
    subprocess.run(['git','clone',URL,str(REPO)],check=True)
else:
    dirty = subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
    if dirty:
        raise RuntimeError('Existing checkout has local changes; save them before updating. Nothing was overwritten.')
    subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
PY = sys.executable  # Colab images may not include the python3-venv system package.
subprocess.run([PY,'-m','pip','install','-r',str(REPO/'requirements.txt')],check=True)
def run(*args):
    subprocess.run([PY,*map(str,args)],cwd=REPO,check=True)
run('-m','unittest','discover','-s','tests','-v')


## Configure and restore checkpoints
Set `RUN_TAG` to a new value when changing input datasets. A parameter-derived key separates different configurations. Data/config mismatches cause an explicit error.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AmazonML2026')
SAMPLE_SIZE = 20000
EDA_MAX_ROWS = 10000  # 0 = full streaming EDA; 10000 = labeled prefix sample.
LIGHTGBM_DEVICE = 'cpu'  # Optional 'gpu'; unsupported backend falls back to CPU.
RUN_TAG = 'eda_integrity_v2'
RUN_TEST = False
TEAM = 'YOUR_TEAM'
cfg = json.loads((REPO/'configs/baseline.json').read_text())
cfg['sample_size'] = SAMPLE_SIZE
cfg['blocking']['final_k'] = 20
cfg['model']['device_type'] = LIGHTGBM_DEVICE
# Optional: cfg['transliteration']['enabled'] = False
# Optional: cfg['normalization']['ambiguous_st_to_street'] = True
key = RUN_TAG + '_' + hashlib.sha256(json.dumps(cfg,sort_keys=True).encode()).hexdigest()[:10]
LOCAL = Path('/content/er_hybrid')/key
LOCAL.mkdir(parents=True,exist_ok=True)
ART = LOCAL/'artifacts'
SAVED = DRIVE_ROOT/'artifacts'/key
if SAVED.exists() and not ART.exists():
    shutil.copytree(SAVED,ART)
ART.mkdir(exist_ok=True)
CONFIG = LOCAL/'config.json'
CONFIG.write_text(json.dumps(cfg,indent=2))
DATA = LOCAL/'dataset'
INDEX = ART/'index_train'
WORK = ART/'run'
FINAL = ART/'final_model'
TEST_INDEX = ART/'index_test'
OUTPUT = ART/'output'
def checkpoint():
    shutil.copytree(ART,SAVED,dirs_exist_ok=True,ignore=shutil.ignore_patterns('*.tmp','*.tmp.npz','*.building*','train_X.npy','train_y.npy'))
def phase(*commands):
    try:
        for command in commands: run(*command)
    finally:
        checkpoint()
print('Free local disk GB:',round(shutil.disk_usage('/content').free/1e9,2))
print('Local artifacts:',ART)
print('Drive checkpoint:',SAVED)


## Prepare files, prove integrity, then run EDA
The manifest proves byte preservation. Integrity records recoverable row repairs and stops on duplicate or unknown IDs before EDA begins.

In [ ]:
expected = [f'train_source{i}.tsv' for i in (1,2,3)] + ['train_ground_truth.tsv']
if not all((DATA/'train'/name).exists() for name in expected):
    run('src/prepare_data.py','--input',DRIVE_ROOT/'input/train','--output',DATA/'train','--split','train')
phase(['src/data_integrity.py','--data',DATA/'train','--split','train','--output',ART/'data_integrity.json'])
print(json.dumps(json.loads((ART/'data_integrity.json').read_text()),indent=2,ensure_ascii=False))
phase(['src/eda.py','--data',DATA/'train','--config',CONFIG,'--max-rows',EDA_MAX_ROWS,'--output',ART/'eda'])
print(json.dumps(json.loads((ART/'eda/report.json').read_text()),indent=2,ensure_ascii=False))
